# Day 020 Project — Second Brain Chatbot

Build a `SecondBrainChatbot` that wraps the RAG pipeline you built in Exercises 1–5 into a usable Q&A assistant over your personal knowledge base.

## Project Requirements

1. Define at least **3 documents** as string constants (your knowledge base)
2. Implement `SecondBrainChatbot` with `__init__(docs, model, chunk_size, overlap)` and `ask(question) -> dict`
3. Ask at least **3 scripted questions** and print each answer with sources
4. Define at least **3 `TestCase` objects** and run a mini eval with `contains_any` scoring
5. Run the checks below to verify correctness


In [ ]:
import re
import ollama
import chromadb
from dataclasses import dataclass, field


## Provided: RAG Pipeline

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    if step <= 0:
        step = 1
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def embed_text(text: str, model: str = "nomic-embed-text") -> list[float]:
    return ollama.embeddings(model=model, prompt=text)["embedding"]


def build_index(
    docs: dict,
    collection_name: str = "second_brain",
    chunk_size: int = 300,
    overlap: int = 50,
):
    client = chromadb.Client()
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    for source, text in docs.items():
        chunks = chunk_text(text, chunk_size, overlap)
        ids, embeddings, documents, metadatas = [], [], [], []
        for i, chunk in enumerate(chunks):
            ids.append(f"{source}__{i}")
            embeddings.append(embed_text(chunk))
            documents.append(chunk)
            metadatas.append({"source": source, "chunk_index": i})
        if ids:
            collection.add(
                ids=ids, embeddings=embeddings,
                documents=documents, metadatas=metadatas,
            )
    return collection


def retrieve(query: str, collection, n_results: int = 3) -> list[dict]:
    if collection.count() == 0:
        return []
    emb = embed_text(query)
    actual_n = min(n_results, collection.count())
    results = collection.query(query_embeddings=[emb], n_results=actual_n)
    return [
        {"text": doc, "source": meta["source"], "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def build_cited_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[{i+1}] Source: {c['source']}\n{c['text']}"
        for i, c in enumerate(chunks)
    )
    return f"Context:\n{context}\n\nQuestion: {question}"


RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer questions using ONLY the "
    "provided context. Cite the source numbers you used (e.g. [1], [2]). "
    "If the answer is not in the context, say 'I don't know.'"
)


def rag_answer(
    question: str,
    collection,
    model: str = "llama3.2",
    n_results: int = 3,
) -> dict:
    chunks = retrieve(question, collection, n_results)
    if not chunks:
        return {"answer": "I don't know.", "sources": []}
    prompt = build_cited_prompt(question, chunks)
    response = ollama.chat(model=model, messages=[
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ])
    answer  = response["message"]["content"]
    sources = list(dict.fromkeys(c["source"] for c in chunks))
    return {"answer": answer, "sources": sources}

## Provided: Eval Helpers (Day 19)

In [ ]:
@dataclass
class TestCase:
    question: str
    expected_keywords: list[str] = field(default_factory=list)
    expected_answer: str = ''


def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)


## Your Implementation

Define your documents and implement `SecondBrainChatbot`.

In [ ]:
# --- Your knowledge base ---
DOCS = {
    'doc1.txt': 'Your first document text here...',
    'doc2.txt': 'Your second document text here...',
    'doc3.txt': 'Your third document text here...',
}


class SecondBrainChatbot:
    def __init__(self, docs: dict, model: str = 'llama3.2',
                 chunk_size: int = 300, overlap: int = 50):
        # TODO: store model, build collection with build_index, init history
        pass

    def ask(self, question: str) -> dict:
        # TODO: call rag_answer, append to history, return result
        pass

    def summary(self) -> str:
        # TODO: return a summary string with question count
        pass


## Ask Questions

In [ ]:
# Build chatbot and ask 3 scripted questions
# bot = SecondBrainChatbot(DOCS)
# result = bot.ask('Your question here?')
# print(result['answer'])
# print('Sources:', result['sources'])


## Mini Eval Run

In [ ]:
EVAL_CASES = [
    # TestCase('question', expected_keywords=['keyword1', 'keyword2']),
]

# eval_results = []
# for tc in EVAL_CASES:
#     answer = bot.ask(tc.question)['answer']
#     matched = [kw for kw in tc.expected_keywords if kw.lower() in answer.lower()]
#     passed_check = bool(matched) if tc.expected_keywords else True
#     icon = '✅' if passed_check else '❌'
#     print(f'{icon} {tc.question[:60]}')
#     if matched: print(f'   Keywords: {matched}')
#     eval_results.append(passed_check)
# pass_rate = sum(eval_results) / len(eval_results) if eval_results else 0.0
# print(f'\nEval pass rate: {pass_rate*100:.1f}%')


## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: SecondBrainChatbot class defined with required methods
    try:
        assert 'SecondBrainChatbot' in globals()
        assert hasattr(SecondBrainChatbot, 'ask')
        assert hasattr(SecondBrainChatbot, 'summary')
        passed += 1; print('✅ Check 1: SecondBrainChatbot class defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: DOCS has at least 3 documents
    try:
        assert 'DOCS' in globals(), 'DOCS not defined'
        assert len(DOCS) >= 3, f'Need >= 3 docs, got {len(DOCS)}'
        passed += 1; print(f'✅ Check 2: DOCS has {len(DOCS)} documents')
    except Exception as e:
        print(f'❌ Check 2: DOCS — {e}')

    # Check 3: chatbot builds and ask() works (embed + LLM call)
    try:
        _bot = SecondBrainChatbot(DOCS)
        assert _bot is not None
        _result = _bot.ask(list(DOCS.keys())[0])  # ask a question about first doc name
        assert isinstance(_result, dict), f'ask() must return dict, got {type(_result)}'
        assert 'answer' in _result and 'sources' in _result
        passed += 1; print('✅ Check 3: chatbot builds and ask() returns valid dict')
    except Exception as e:
        print(f'❌ Check 3: chatbot.ask() — {e}')

    # Check 4: EVAL_CASES has at least 3 cases
    try:
        assert 'EVAL_CASES' in globals(), 'EVAL_CASES not defined'
        assert len(EVAL_CASES) >= 3, f'Need >= 3 eval cases, got {len(EVAL_CASES)}'
        passed += 1; print(f'✅ Check 4: EVAL_CASES has {len(EVAL_CASES)} cases')
    except Exception as e:
        print(f'❌ Check 4: EVAL_CASES — {e}')

    # Check 5: history grows with each ask()
    try:
        _b2 = SecondBrainChatbot({'note.txt': 'Python is a programming language.'})
        _b2.ask('What is Python?')
        _b2.ask('Tell me more.')
        assert len(_b2._history) == 2, f'expected 2 history entries, got {len(_b2._history)}'
        passed += 1; print('✅ Check 5: history accumulates with each ask()')
    except Exception as e:
        print(f'❌ Check 5: history — {e}')

    if passed == total:
        print('🎉 Project complete!')
    print(f'\nScore: {passed}/{total}')

_run_project_checks()


## Bonus Challenges

- Use ChromaDB's `PersistentClient` so the index survives between sessions
- Add token tracking from Day 18: include `UsageTracker` in `SecondBrainChatbot` and print total token usage in `summary()`
- Filter low-quality retrieved chunks by `distance` threshold before building the prompt
- Add an `llm_judge` eval run (Day 19) to score answer quality alongside keyword checks
- Try different `chunk_size`/`overlap` values and observe how retrieval quality changes